In [135]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [136]:
from pypots.data.generating import gene_physionet2012
from pypots.utils.random import set_random_seed

In [ ]:
!pip 

In [137]:
tsdb.list()

['physionet_2012',
 'physionet_2019',
 'electricity_load_diagrams',
 'beijing_multisite_air_quality',
 'italy_air_quality',
 'vessel_ais',
 'electricity_transformer_temperature',
 'pems_traffic',
 'solar_alabama',
 'ucr_uea_ACSF1',
 'ucr_uea_Adiac',
 'ucr_uea_AllGestureWiimoteX',
 'ucr_uea_AllGestureWiimoteY',
 'ucr_uea_AllGestureWiimoteZ',
 'ucr_uea_ArrowHead',
 'ucr_uea_Beef',
 'ucr_uea_BeetleFly',
 'ucr_uea_BirdChicken',
 'ucr_uea_BME',
 'ucr_uea_Car',
 'ucr_uea_CBF',
 'ucr_uea_Chinatown',
 'ucr_uea_ChlorineConcentration',
 'ucr_uea_CinCECGTorso',
 'ucr_uea_Coffee',
 'ucr_uea_Computers',
 'ucr_uea_CricketX',
 'ucr_uea_CricketY',
 'ucr_uea_CricketZ',
 'ucr_uea_Crop',
 'ucr_uea_DiatomSizeReduction',
 'ucr_uea_DistalPhalanxOutlineCorrect',
 'ucr_uea_DistalPhalanxOutlineAgeGroup',
 'ucr_uea_DistalPhalanxTW',
 'ucr_uea_DodgerLoopDay',
 'ucr_uea_DodgerLoopGame',
 'ucr_uea_DodgerLoopWeekend',
 'ucr_uea_Earthquakes',
 'ucr_uea_ECG200',
 'ucr_uea_ECG5000',
 'ucr_uea_ECGFiveDays',
 'ucr_uea_E

In [142]:
data = tsdb.load('physionet_2012')

2025-03-04 15:05:12 [INFO]: You're using dataset physionet_2012, please cite it properly in your work. You can find its reference information at the below link: 
https://github.com/WenjieDu/TSDB/tree/main/dataset_profiles/physionet_2012
2025-03-04 15:05:12 [INFO]: Dataset physionet_2012 has already been downloaded. Processing directly...
2025-03-04 15:05:12 [INFO]: Dataset physionet_2012 has already been cached. Loading from cache directly...
2025-03-04 15:05:12 [ERROR]: ❌ Loading data failed. Operation aborted. Investigate the error below:
No module named 'pandas.core.indexes.numeric'
2025-03-04 15:05:12 [INFO]: Loaded successfully!


In [140]:
data['training_setA']

,AST,Age,Alkalinephos,BUN,BaseExcess,Bilirubin_direct,Bilirubin_total,Calcium,Chloride,Creatinine,...,Resp,SBP,SaO2,SepsisLabel,Temp,TroponinI,Unit1,Unit2,WBC,pH
0,NaN,77.27,NaN,53.0,1.0,NaN,NaN,NaN,111.0,2.1,...,13.5,121.00,77.0,0,36.50,NaN,0.0,1.0,9.9,7.40
1,NaN,77.27,NaN,53.0,1.0,NaN,NaN,NaN,111.0,NaN,...,12.0,113.25,NaN,0,36.25,NaN,0.0,1.0,9.9,7.40
2,NaN,77.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.0,132.75,NaN,0,36.25,NaN,0.0,1.0,NaN,NaN
3,NaN,77.27,NaN,NaN,-3.0,NaN,NaN,NaN,NaN,NaN,...,12.0,103.50,NaN,0,36.10,NaN,0.0,1.0,NaN,7.34
4,NaN,77.27,NaN,NaN,-3.0,NaN,NaN,NaN,NaN,NaN,...,12.5,128.75,NaN,0,36.00,NaN,0.0,1.0,NaN,7.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,NaN,66.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.0,121.00,NaN,0,NaN,NaN,0.0,1.0,NaN,NaN
10,NaN,66.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,112.00,NaN,0,NaN,NaN,0.0,1.0,NaN,NaN
11,NaN,66.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,102.00,NaN,0,NaN,NaN,0.0,1.0,NaN,NaN
12,NaN,66.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,86.00,NaN,0,36.11,NaN,0.0,1.0,NaN,NaN


In [ ]:
data['X']['patient'] = (data['X'].Time==1).cumsum()

In [ ]:
import pandas as pd

# Assuming your DataFrame is named `data` and has columns 'patient', 'Time', 'X'
# Drop duplicate entries (or use .agg() if you prefer an aggregation)
data_unique = data['X'].drop_duplicates(subset=['patient', 'Time'])
#data_agg = data.groupby(['patient', 'Time'], as_index=False).agg({'X': 'mean'})


# Create a MultiIndex for all patients and time points 1 through 48
all_patients = data_unique['patient'].unique()
full_index = pd.MultiIndex.from_product([all_patients, range(1, 49)],
                                          names=['patient', 'Time'])

# Reindex the DataFrame
data_complete = data_unique.set_index(['patient', 'Time']).reindex(full_index).reset_index()

In [ ]:
import numpy as np
from pygrinder import mcar
import torch

X = data_complete.values[:,2:]
relevant_feats = (np.isnan(X).mean(axis=0) < 0.95)
X = X[:,relevant_feats]

## Normalize_data
mean, std = np.nanmean(X,axis=0)[None,:], np.nanstd(X, axis=0)[None,:]
std[std==0] = 1
X_normalized = (X - mean) / std
X_normalized = X_normalized.clip(min=-2,max=2)

# reshape by patient
X_normalized = torch.tensor(X_normalized.reshape(-1,48,X_normalized.shape[1])).float()[:500]

X = mcar(torch.clone(X_normalized), p = 0.1)

In [ ]:
dataset = {'X' : X}
len_dataset = len(dataset['X'])
cut = int(len_dataset*0.7)
dataset_train, dataset_val = {'X':dataset['X'][:cut]}, {'X':dataset['X'][cut:], 'X_ori':X_normalized[cut:]}

In [ ]:
X_normalized.shape

In [ ]:
plt.plot(X_normalized[0])

In [ ]:
from pypots.optim.adam import Adam
from torch.optim.lr_scheduler import LRScheduler

from pypots.imputation import GP_VAE

gpvae = GP_VAE(n_steps = dataset_train['X'].shape[1], 
            n_features = dataset_train['X'].shape[2], 
            latent_size = 30, 
            epochs = 500, 
            batch_size = 32,
            beta = 1., 
            K = 5,  
            encoder_sizes = (64,32), 
            decoder_sizes = (64,32),
            optimizer = Adam(weight_decay=0.001), #lr = 1e-4
            patience = 30
            )

gpvae.model.backbone.alpha = 1e-1
gpvae.model.backbone.beta = 1e-2
gpvae.model.backbone.gamma = 5 * 1e-5
gpvae.model.backbone.noise_sigma = 1e-3
gpvae.model.backbone.sampling = True
gpvae.model.backbone.use_mean = True
gpvae.model.backbone.noise_std = 1e-2
gpvae.model.backbone.detach = True
gpvae.model.backbone.compensate = False
gpvae.use_gp = False
gpvae.train_gp = False
gpvae.gp.plot_while_training = True

gpvae.model.backbone.p = 0.3
gpvae.model.backbone.device = 'cpu'

In [ ]:
gpvae.fit(dataset_train, dataset_val)

In [ ]:
gpvae.fit(dataset_train, dataset_val)

In [ ]:
gpvae.gp.plot_while_training = True
gpvae.fit_kernel(dataset_train, dataset_val, training_iter = 50)

In [ ]:
torch.save(gpvae.model.backbone.encoder.state_dict().copy(), 'encoder')
torch.save(gpvae.model.backbone.decoder.state_dict().copy(), 'decoder')

In [ ]:
state_dict = torch.load('encoder');
gpvae.model.backbone.encoder.load_state_dict(state_dict);
state_dict = torch.load('decoder');
gpvae.model.backbone.decoder.load_state_dict(state_dict);

In [ ]:
X_val_imputed = gpvae.impute(dataset_val, with_gp = False)
plt.plot(X_val_imputed[0]);
plt.gca().set_prop_cycle(None)
plt.plot(dataset_val['X'][0],'o');

In [ ]:
mse_obs, mse_hidden = plot_reconstruction_with_GP(X[::5], X_normalized[::5], gpvae, i = 4)